**Parallel Chains**

In [13]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda
load_dotenv()

from langchain_openai import ChatOpenAI

llm_openai = ChatOpenAI(model="gpt-3.5-turbo", api_key=os.environ.get("OPENAI_API_KEY"),base_url="https://openrouter.ai/api/v1",temperature=0.9)

In [14]:
# Task - 1 : Prompt
prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a movie summarizer"),
    ("human","Please summarize the movie in brief : {input}")
])

In [15]:
# Task - 2 : LLM
llm_openai = ChatOpenAI(model="gpt-3.5-turbo", api_key=os.environ.get("OPENAI_API_KEY"),base_url="https://openrouter.ai/api/v1",temperature=0.9)

In [16]:
# Task - 3 : Parser
str_parser = StrOutputParser()

In [17]:
# Task - 4 : Custom Function
from langchain_core.runnables import RunnableLambda
def dictionary_maker(text: str)-> dict:

    return {"text":text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

**Chain 1**

In [18]:
# Task 1 : Prompt
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a linkedin post generator"),
    ("human","Create a post for following text for LinkedIn : {text}")
])

In [19]:
# Task 2 : LLM
llm_openai = ChatOpenAI(model="gpt-3.5-turbo", api_key=os.environ.get("OPENAI_API_KEY"),base_url="https://openrouter.ai/api/v1",temperature=0.9)

In [20]:
# Task 3 : String output parser
str_parser = StrOutputParser()

In [21]:
chain_linkedin = linkedin_prompt | llm_openai | str_parser

**Chain 2 - as afunction**

In [22]:
def insta_chain(text:dict):

    text = text["text"]
    # Task 1 : Prompt
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system","You are a Instagram post generator"),
        ("human","Create a post for following text for Instagram : {text}")
    ])
    
    # Task 2 : LLM
    llm_openai = ChatOpenAI(model="gpt-3.5-turbo", api_key=os.environ.get("OPENAI_API_KEY"),base_url="https://openrouter.ai/api/v1",temperature=0.9)

    # Task 3 : String Output Parser
    str_parser = StrOutputParser()

    # Task 4 : Chain
    chain_Instagram = insta_prompt | llm_openai | str_parser

    result=chain_Instagram.invoke(text)
    return result

insta_chain_runnable = RunnableLambda(insta_chain)

**Final Orchestration**

In [23]:
Final_chain = prompt_template | llm_openai | str_parser | dictionary_maker_runnable | RunnableParallel(branches = {"linkedin":chain_linkedin, "Instagram":insta_chain_runnable})
Final_chain.invoke("Dil Chahta hai")

{'branches': {'linkedin': '🎬 "Dil Chahta Hai" - A cinematic masterpiece that beautifully captures the essence of friendship, love, and personal growth. 🌟 Set against the scenic backdrops of Goa and Sydney, this Bollywood film follows the journey of three young men as they navigate through the complexities of relationships and career choices. 💼💔 Through moments of joy, heartbreak, and self-discovery, the characters learn valuable lessons about life and relationships. 🌿🌺 Let us all cherish the timeless beauty and profound insights of this classic movie. 🎥✨ #DilChahtaHai #Friendship #Love #PersonalGrowth #Bollywood #Cinema #LifeLessons 🌟🎬',
  'Instagram': '🎬✨ "Dil Chahta Hai" 🌟💫\n\nStep into the world of friendship, love, and self-discovery with this Bollywood gem! 🌸🌿 Follow the journey of three young men as they navigate through the ups and downs of life, relationships, and adulthood. Set against the vibrant backdrops of Goa and Sydney, this film beautifully captures the essence of frien